# DS2002 · JSON and Nested Records

**Lecture — 2026-09-28 · Fall 2026**  
**Class time:** 45 minutes

---

## What JSON is

JSON stands for JavaScript Object Notation. Ignore the JavaScript part. It started there around 2001 and every language uses it now.

It is a way of writing structured data as plain text. That is the entire idea. A JSON file is text you can open in Notepad and read with your eyes — nothing compressed, nothing binary, no special software required.

You have been staring at JSON all semester without being told. Open any `.ipynb` file in a text editor and that is JSON: your cells, your outputs, and the metadata that makes notebook merge conflicts so ugly in Git.

JSON is built out of exactly two containers.

An **object** is a set of name-value pairs in curly braces. Python calls this a dictionary.

```json
{"vendor_id": "V-01", "name": "Hoos Burgers"}
```

An **array** is an ordered list in square brackets. Python calls this a list.

```json
["Cheeseburger", "Hot Dog", "Rain Poncho"]
```

Everything else is those two nested inside each other, as deep as whoever built the thing felt like going.

### Why anybody uses it

You already know a format for data. So why is the entire web running on a different one?

**A CSV can only describe a rectangle.** Rows and columns, every row the same width. That is fine for a spreadsheet and it falls apart the moment one thing contains several other things. One order has three line items. One vendor covers four zones. To force that into a CSV you either repeat the order header on every line or you invent a second file and a key to join them. JSON just puts the list inside the order.

**The names travel with the data.** In a CSV, column three is the price only because a header row said so. Lose the header or reorder the columns and nothing warns you. In JSON every value is labeled where it sits, so a program can read it without being told the layout in advance.

**Every language reads it.** Python, R, JavaScript, Java, Go — one function call in all of them. That matters when the service sending you data was written by somebody else in a language you do not use.

**It is just text.** It travels over the same HTTP that carries web pages, it sits in log files, you can paste it into an email. When something breaks you can look at it and see what happened.

The trade is nesting. You get a format that matches the real shape of things, and in exchange you cannot hand it to a chart or a `groupby` until you flatten it. Flattening is the rest of today.

### The rules

JSON is a small language. There are six kinds of value and that is the whole list:

| Type | Looks like |
|---|---|
| string | `"Hoos Burgers"` |
| number | `320` or `4.6` |
| boolean | `true` or `false` |
| null | `null` |
| object | `{"a": 1}` |
| array | `[1, 2, 3]` |

The syntax rules are just as short. Break any one of them and the file will not parse at all — JSON has no partial credit.

1. **Strings use double quotes.** Never single. `'Hoos Burgers'` is not JSON.
2. **Keys are strings too**, so they get double quotes as well. `{name: "x"}` is invalid, `{"name": "x"}` is valid.
3. **`true`, `false`, and `null` are lowercase.** Python writes `True` and `None`. JSON does not accept those.
4. **No comma after the last item.** `{"a": 1,}` is an error.
5. **No comments.** There is no way to leave a note inside a JSON file.
6. **Whitespace does not matter.** Line breaks and indenting are for humans.

Two things JSON knows nothing about, which will cost you later: **dates** and **money**. A date is just a string, so `"2026-09-28"` and `"09/28/2026"` are equally valid and equally meaningless to the parser. Converting them is your job, using the same `pd.to_datetime` from last week.

### JSON is text. A dictionary is not.

This one trips up almost everybody, because the two look nearly identical on the page.

A JSON document is a **string**. Before you can pull anything out of it, you have to parse it into real Python objects. `json.loads` does that, and the name means "load from string."

In [ ]:
import json

# This is JSON: a string, exactly as it would arrive from a web server.
raw = '''
{
  "vendor_id": "V-01",
  "name": "Hoos Burgers",
  "open": true,
  "manager": null,
  "rating": 4.6,
  "items": ["Cheeseburger", "Hot Dog"]
}
'''

print('what we received :', type(raw))

data = json.loads(raw)          # loads = load from string
print('after json.loads :', type(data))
print()
print('name    :', data['name'])
print('items   :', data['items'], type(data['items']))
print('open    :', data['open'], type(data['open']))
print('manager :', data['manager'], type(data['manager']))

Once it is parsed you index it like any Python dictionary, because it *is* one now. The translation is fixed:

| JSON | becomes in Python |
|---|---|
| object | `dict` |
| array | `list` |
| string | `str` |
| number | `int` or `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

Look at what came back for `"open": true` and `"manager": null` above. Python's `True` and `None`. You did not ask for that conversion and you cannot turn it off.

It works in reverse too. `json.dumps` — dump to string — takes Python objects and writes JSON text, putting `true` and `null` back the way JSON spells them. That is what you use to send data somewhere or save it to a file.

In [ ]:
back = json.dumps(data, indent=2)

print(type(back))
print(back)

### When it will not parse

Four ways to write something that is almost JSON. Three of them are habits carried over from Python.

In [ ]:
attempts = {
    'single quotes': "{'name': 'Hoos Burgers'}",
    'trailing comma': '{"name": "Hoos Burgers",}',
    'Python True': '{"open": True}',
    'unquoted key': '{name: "Hoos Burgers"}',
    'valid JSON': '{"name": "Hoos Burgers"}',
}

for label, text in attempts.items():
    try:
        json.loads(text)
        print(f'{label:16} OK')
    except json.JSONDecodeError as err:
        print(f'{label:16} FAILED at column {err.colno}: {err.msg}')

Read those messages, because you will see them again. The parser tells you the exact column where it gave up, and the message usually names the rule you broke. "Expecting property name enclosed in double quotes" means a key is unquoted or in single quotes.

One piece of advice for when this happens to you on real data from an API: do not start editing the text by hand. Print the first 200 characters first. Nine times out of ten the server sent back an HTML error page instead of JSON, and the actual problem is your request, not your parsing.

---

## Data that arrives in the wrong shape

Now the part that costs you time. Everything you have cleaned so far came as a rectangle. Web data does not. What you get back is nested: dictionaries inside lists inside dictionaries, as deep as whoever designed the API felt like going.

JSON is shaped that way for good reasons — it mirrors how the data is actually structured, and it avoids repeating the same values on every row. But you cannot group, join, or plot a nested dictionary. So the job this week is **flattening**: turning nesting into one row per thing you want to count.

That last phrase is the whole skill. Before you write any code, answer one question: *what is one row of my result?* One vendor? One order? One line item on one order? The answer determines every argument you pass.

### Shape 1 — a list of flat records

The easy case, and the one you will wish for. `pd.DataFrame` handles it directly.

In [ ]:
import json, pandas as pd

flat = [
    {'vendor_id': 'V-01', 'name': 'Hoos Burgers', 'orders': 320},
    {'vendor_id': 'V-10', 'name': 'Cav Merch',    'orders': 110},
]
pd.DataFrame(flat)

### Shape 2 — nested dictionaries

Now the values are grouped under a sub-object. `pd.DataFrame` leaves the whole dictionary sitting in a cell, which is useless. `json_normalize` flattens it and names the columns with dots.

In [ ]:
nested = [
    {'vendor_id': 'V-01', 'name': 'Hoos Burgers',
     'sales': {'orders': 320, 'revenue': 2400}},
    {'vendor_id': 'V-10', 'name': 'Cav Merch',
     'sales': {'orders': 110, 'revenue': 2600}},
]

print('--- pd.DataFrame leaves the dict in the cell ---')
print(pd.DataFrame(nested))
print()
print('--- json_normalize flattens it ---')
pd.json_normalize(nested)

Dots in column names are legal but awkward — `df.sales.revenue` will not work, you need `df['sales.revenue']`. Change the separator if you would rather have underscores.

In [ ]:
pd.json_normalize(nested, sep='_')

### Shape 3 — records buried under a wrapper

This is what a real API response looks like: some metadata at the top, and the rows you want one level down. `record_path` says which list holds the rows; `meta` says which top-level fields to copy onto every row.

In [ ]:
payload = {
    'game': 'UVA vs Duke',
    'date': '2026-10-10',
    'vendors': [
        {'vendor_id': 'V-01', 'name': 'Hoos Burgers',
         'sales': {'orders': 320, 'revenue': 2400}},
        {'vendor_id': 'V-10', 'name': 'Cav Merch',
         'sales': {'orders': 110, 'revenue': 2600}},
    ],
}
print(json.dumps(payload, indent=2))

In [ ]:
# Wrong: loses the game and date entirely
print(pd.json_normalize(payload['vendors']).columns.tolist())
print()
# Right: record_path picks the rows, meta carries the context down
pd.json_normalize(payload, record_path='vendors', meta=['game', 'date'])

Without `meta` you would have a table of vendors with no idea which game they belong to. The moment you concatenate two games together, that context is the only thing keeping the rows distinguishable.

### Shape 4 — a list inside a list

Two levels of nesting: zones, and vendors within each zone. `record_path` takes a list of keys to walk down, and `meta` can reach into intermediate levels with its own nested list.

In [ ]:
day = {
    'date': '2026-10-10',
    'zones': [
        {'zone': 'A', 'capacity': 400,
         'vendors': [{'name': 'Hoos Burgers', 'orders': 300},
                     {'name': 'Stadium Dogs', 'orders': 210}]},
        {'zone': 'B', 'capacity': 150,
         'vendors': [{'name': 'Wahoo Wings', 'orders': 150}]},
    ],
}

vendors = pd.json_normalize(
    day,
    record_path=['zones', 'vendors'],
    meta=['date', ['zones', 'zone'], ['zones', 'capacity']],
)
vendors

One row per vendor, with the date and the zone's own fields carried down. Note that the meta columns keep their full path as the name — rename them, because `zones.zone` is not a column name you want to type twenty more times.

In [ ]:
vendors = vendors.rename(columns={'zones.zone': 'zone',
                                 'zones.capacity': 'zone_capacity'})
vendors.groupby('zone')['orders'].sum()

### explode vs normalize

Two tools that look interchangeable and are not.

- **`json_normalize`** works on the raw JSON, before it is a DataFrame.
- **`explode`** works on a DataFrame that already has a list sitting in a column.

You reach for `explode` when the nesting survived into your frame — often because the data came from a CSV where somebody stored a list as a string, or because you loaded it with `pd.DataFrame` instead of normalizing.

In [ ]:
orders = pd.DataFrame({
    'order_id': [1, 2],
    'items': [['Cheeseburger', 'Soda'], ['Foam Finger']],
})
print('--- lists trapped in a column ---')
print(orders)
print()
print('--- after explode: one row per item ---')
orders.explode('items').reset_index(drop=True)

### Where this breaks

Real API responses are ragged. Not every record has every field, and pretending otherwise is how a pipeline dies at 2am. Watch what `json_normalize` does with a missing key.

In [ ]:
ragged = [
    {'vendor_id': 'V-01', 'sales': {'orders': 320, 'revenue': 2400}},
    {'vendor_id': 'V-10', 'sales': {'orders': 110}},          # no revenue
    {'vendor_id': 'V-18'},                                    # no sales at all
]
out = pd.json_normalize(ragged)
print(out)
print()
print(out.isnull().sum())

It does the reasonable thing: missing fields become `NaN`. That is good behavior and a trap at the same time, because `NaN` here means "the API did not tell us," and if you call `.fillna(0)` you have just invented a revenue figure of zero for a vendor who may have made two thousand dollars.

The other failure is louder. Ask for a `record_path` that some records do not have and it raises rather than guessing:

In [ ]:
try:
    pd.json_normalize(ragged, record_path='lines')
except Exception as e:
    print(type(e).__name__, '->', e)

Loud failure is the better outcome. Compare it to the alternative: silently getting an empty frame and charting nothing.

**The habit for this week:** after every normalize, check the row count against what you expected and check `isnull().sum()`. Nested data hides missing values far better than a CSV does.

### Practice 1 — one row per line item

This is the shape you will need on Friday. Flatten to one row per **line item**, keeping the order id and the vendor.

*Expected: 4 rows.*

In [ ]:
receipt = {
    'game': 'UVA vs Duke',
    'orders': [
        {'order_id': 1, 'vendor': 'Hoos Burgers',
         'lines': [{'item': 'Cheeseburger', 'qty': 2},
                   {'item': 'Soda', 'qty': 1}]},
        {'order_id': 2, 'vendor': 'Cav Merch',
         'lines': [{'item': 'Foam Finger', 'qty': 1}]},
        {'order_id': 3, 'vendor': 'Hoos Burgers',
         'lines': [{'item': 'Hot Dog', 'qty': 3}]},
    ],
}

# TODO: one row per line item, with order_id, vendor, and game on every row

### Practice 2 — validate, then answer

Check your row count, then print total quantity per vendor.

In [ ]:
# TODO: assert the row count, then group

### Practice 3 — the missing-value judgment call

Go back to the `ragged` list. Produce a frame where a vendor with no reported revenue is clearly distinguishable from a vendor with zero revenue. Print it and say in a comment how a reader can tell the difference.

In [ ]:
# TODO